# EduPredict: Exploratory Data Analysis (Phase 2)

This notebook performs an in-depth exploratory analysis of the UCI Student Performance Portuguese dataset (`student-por.csv`).

### Project Objective:
Predict student academic performance mapped to four distinct grade categories:
- **Class 0 (Poor):** Final Grade `0–9`
- **Class 1 (Average):** Final Grade `10–12`
- **Class 2 (Good):** Final Grade `13–15`
- **Class 3 (Excellent):** Final Grade `16–20`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

CLASS_PALETTE = {
    'Poor (0)': '#e74c3c',
    'Average (1)': '#f39c12',
    'Good (2)': '#3498db',
    'Excellent (3)': '#2ecc71'
}

# Load dataset
df = pd.read_csv('../data/student-por.csv', sep=';')
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Data Integrity & Missing Values Check

In [ ]:
print(f"Total Missing Values: {df.isnull().sum().sum()}")
print(f"Total Duplicated Rows: {df.duplicated().sum()}")
df.info()

## 2. Target Variable (G3) & 4-Class Distribution

In [ ]:
bins = [-1, 9, 12, 15, 20]
labels = ['Poor (0)', 'Average (1)', 'Good (2)', 'Excellent (3)']
df['performance_class'] = pd.cut(df['G3'], bins=bins, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['G3'], kde=True, bins=21, ax=axes[0], color='#2980b9')
axes[0].set_title('Distribution of Continuous Final Grades (G3)')

class_counts = df['performance_class'].value_counts()[labels]
axes[1].bar(labels, class_counts.values, color=[CLASS_PALETTE[l] for l in labels])
axes[1].set_title('Target 4-Class Breakdown')
for i, v in enumerate(class_counts.values):
    axes[1].text(i, v + 3, f"{v} ({v/len(df):.1%})", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Period Grade Progression (G1 & G2 vs G3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=df, x='G1', y='G3', ax=axes[0], scatter_kws={'alpha': 0.5, 'color': '#3498db'}, line_kws={'color': '#e74c3c'})
axes[0].set_title(f"G1 vs G3 (r = {df['G1'].corr(df['G3']):.3f})")

sns.regplot(data=df, x='G2', y='G3', ax=axes[1], scatter_kws={'alpha': 0.5, 'color': '#2ecc71'}, line_kws={'color': '#e74c3c'})
axes[1].set_title(f"G2 vs G3 (r = {df['G2'].corr(df['G3']):.3f})")
plt.tight_layout()
plt.show()

## 4. Key Factors Influencing Academic Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Study Time
sns.boxplot(data=df, x='studytime', y='G3', ax=axes[0, 0], palette='Blues', hue='studytime', legend=False)
axes[0, 0].set_title('G3 by Study Time (1: <2h, 2: 2-5h, 3: 5-10h, 4: >10h)')

# Past Failures
sns.boxplot(data=df, x='failures', y='G3', ax=axes[0, 1], palette='Reds_r', hue='failures', legend=False)
axes[0, 1].set_title('G3 by Past Class Failures (0 to 3)')

# School Absences
sns.scatterplot(data=df, x='absences', y='G3', hue='performance_class', palette=CLASS_PALETTE, ax=axes[1, 0])
axes[1, 0].set_title('G3 by School Absences')

# Home Internet
sns.boxplot(data=df, x='internet', y='G3', ax=axes[1, 1], palette='Pastel1', hue='internet', legend=False)
axes[1, 1].set_title('G3 by Home Internet Access')

plt.tight_layout()
plt.show()

## 5. Numerical Feature Correlation Matrix

In [ ]:
num_cols = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
plt.figure(figsize=(12, 9))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.4, vmax=1.0, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Attributes')
plt.show()